In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


# Sklearn Config

In [3]:
from sklearn import set_config

set_config("pandas")

# Import Data

In [4]:
kagglehub.competition_download("playground-series-s6e9")

'/kaggle/input/competitions/playground-series-s6e9'

In [5]:
base = "../input/competitions/playground-series-s6e9"

train_path = f"{base}/train.csv"
test_path = f"{base}/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Explore Data

In [6]:
def prepare_X(dataset: pd.DataFrame):
    return dataset.drop(["id", "Will_Buy_EV"], axis=1)


def prepare_y(dataset: pd.DataFrame):
    return (dataset["Will_Buy_EV"] == "Yes").astype("int")


train_X = prepare_X(train)
train_y = prepare_y(train)

In [7]:
who_will_buy = train[train["Will_Buy_EV"] == "Yes"]

who_will_buy.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
6,6,55,109219.0,42.9,1,1,5,4.0,Male,Suburban,Sedan,Yes,Yes,Low,Yes
9,9,57,114827.0,66.9,2,0,1,5.0,Male,Rural,Sedan,Yes,Yes,Low,Yes
13,13,26,83059.0,33.3,1,10,18,5.0,Male,Urban,Sedan,Yes,Yes,Low,Yes
14,14,25,92697.0,19.5,2,6,11,5.0,Male,Urban,Sedan,Yes,Yes,Low,Yes


In [8]:
who_wont_buy = train[train["Will_Buy_EV"] == "No"]
who_wont_buy.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
5,5,50,95938.0,55.1,2,8,15,3.0,Other,Urban,Sedan,No,No,Low,No


# Feature Engineering

## Category feature unification

In [9]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer


def prepare_category_feature_unification_step(unification_features: list = []):   
    unificator = FunctionTransformer(unite_categories_features) if len(unification_features) != 0 else FunctionTransformer()
    unificator_step = (unificator, unification_features)
    result_transformer = make_column_transformer(unificator_step, remainder="passthrough", verbose_feature_names_out=False)
    result_transformer_label = "unification_" + "_".join(unification_features)
    result_transformer_step = (result_transformer_label, result_transformer)
    return result_transformer_step
    
    
def unite_categories_features(dataset):
    united_features = dataset[dataset.columns[0]].astype(str)
    united_feature_name = "Union_" + "_".join(dataset.columns)
    
    for feature in dataset.columns[1:]:
        united_features += " " + dataset[feature].astype(str)
    
    updated_dataset = dataset.copy()
    updated_dataset[united_feature_name] = united_features
    return updated_dataset


## Test feature unification 

In [10]:
def test_feature_unification(test_dataset):
    test_united_feature_name = "Union_" + "_".join(test_dataset.columns)
    test_dataset_after_feature_unification = unite_categories_features(test_dataset)
    test_source_dataset_after_feature_unification(test_united_feature_name, test_dataset)
    test_result_dataset_after_feature_unification(test_united_feature_name, test_dataset_after_feature_unification)


def test_source_dataset_after_feature_unification(test_united_feature_name, source_dataset):
    assertation_message = "Source dataset has modified after feature unification"
    assertation_check = test_united_feature_name not in source_dataset
    assert assertation_check, assertation_message


def test_result_dataset_after_feature_unification(test_united_feature_name, result_dataset):
    assertation_message = "Dataset after feature unification hasn't modified"
    assertation_check = test_united_feature_name in result_dataset
    assert assertation_check, assertation_message


test_dataset = train_X[["Home_Charging_Possible", "Subsidy_Available"]].copy()
test_feature_unification(test_dataset)

## All available charging station feature

In [11]:
def prepare_available_charging_stations_count_step():
    counter = FunctionTransformer(count_available_charging_stations)
    counter_step = (counter, ["Home_Charging_Possible", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"])
    result_transformer_label = "count_available_charging_stations"
    result_transformer = make_column_transformer(counter_step, remainder="passthrough", verbose_feature_names_out=False)
    result_transformer_step = (result_transformer_label, result_transformer)
    return result_transformer_step


def count_available_charging_stations(dataset):
    updated_dataset = dataset.copy()
    there_is_charging_at_home = updated_dataset["Home_Charging_Possible"] == "Yes"
    updated_dataset["available_charging_stations"] = updated_dataset["Charging_Stations_Near_Home"] + updated_dataset["Charging_Stations_Near_Work"] + there_is_charging_at_home
    return updated_dataset

## Using KBinsDiscretizer

In [12]:
from sklearn.preprocessing import KBinsDiscretizer


def prepare_discretize_for_classes_step(feature = None, n_bins = 10, encode="ordinal", strategy="uniform"):
    '''
    The function doesn't modify a source feature, but create new one with name "class_" + feature name 
    '''
    if feature is None:
        return ("identity", FunctionTransformer())

    discretize_function = prepare_discretize_for_classes(n_bins=n_bins, encode=encode, strategy=strategy)
    discretize_transformer = FunctionTransformer(discretize_function)
    discretizer_step = (discretize_transformer, [feature])
    result_transformer = make_column_transformer(discretizer_step, remainder="passthrough", verbose_feature_names_out=False)
    result_transformer_label = "class_" + feature
    result_transformer_step = (result_transformer_label, result_transformer)
    return result_transformer_step


def prepare_discretize_for_classes(n_bins, encode, strategy):
    def discretize_for_classes(dataset):
        k_bins_discretizer = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy)
        updated_dataset = dataset.copy()
        updated_dataset["class_" + updated_dataset.columns[0]] = k_bins_discretizer.fit_transform(updated_dataset)
        return updated_dataset
    return discretize_for_classes
    

# Prepare preprocessing

In [22]:
from sklearn.pipeline import Pipeline


def prepare_preprocessing():
    unification_features = ["Home_Charging_Possible", "Subsidy_Available"]
    home_charging_and_subsidy_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["Subsidy_Available", "Environmental_Concern_Level"]
    subsidy_and_environmental_concern_unification = prepare_category_feature_unification_step(unification_features)

    annual_income_discretizer = prepare_discretize_for_classes_step("Annual_Income_USD")
    age_discretizer = prepare_discretize_for_classes_step("Age")
    daily_commute_discretizer = prepare_discretize_for_classes_step("Daily_Commute_km")

    unification_features = ["class_Annual_Income_USD", "Current_Car_Type"]
    class_annual_income_and_car_type_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_Age", "Gender"]
    class_age_and_gender_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_Daily_Commute_km", "Range_Anxiety_Level"]
    class_daily_commute_and_range_anxiety_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_Annual_Income_USD", "Subsidy_Available", "Environmental_Concern_Level"]
    class_annual_income_and_subsidy_and_environmental_concern_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_Age", "Subsidy_Available", "Environmental_Concern_Level"]
    class_age_and_subsidy_and_environmental_concern_unification = prepare_category_feature_unification_step(unification_features)

    count_available_charging_stations_step = prepare_available_charging_stations_count_step()

    available_charging_stations_discretizer = prepare_discretize_for_classes_step("available_charging_stations")
    
    unification_features = ["class_available_charging_stations", "class_Daily_Commute_km"]
    class_available_charging_stations_and_class_daily_commute_km_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_available_charging_stations", "class_Annual_Income_USD"]
    class_available_charging_stations_and_class_annual_income_usd = prepare_category_feature_unification_step(unification_features)

    unification_features = ["class_Annual_Income_USD", "class_Daily_Commute_km"]
    class_annual_income_usd_and_class_daily_commute_km = prepare_category_feature_unification_step(unification_features)
    
    unification_features = ["Charging_Stations_Near_Home", "Home_Charging_Possible", "Age"]
    сharging_statitions_near_home_and_home_charging_possible_and_age_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["Age", "Current_Car_Type"]
    age_and_car_type_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["Charging_Stations_Near_Home", "Gender"]
    charging_statitions_near_home_and_gender_unification = prepare_category_feature_unification_step(unification_features)

    unification_features = ["Subsidy_Available", "Current_Car_Type"]
    subsidy_and_car_type_unification = prepare_category_feature_unification_step(unification_features)
    
    preprocessing_pipeline = Pipeline(steps=[
        home_charging_and_subsidy_unification,
        subsidy_and_environmental_concern_unification,
        annual_income_discretizer,
        age_discretizer,
        daily_commute_discretizer,
        class_annual_income_and_car_type_unification,
        count_available_charging_stations_step,
        available_charging_stations_discretizer,
        class_available_charging_stations_and_class_annual_income_usd,
        class_annual_income_usd_and_class_daily_commute_km,
        сharging_statitions_near_home_and_home_charging_possible_and_age_unification,
        age_and_car_type_unification,
        charging_statitions_near_home_and_gender_unification,
        subsidy_and_car_type_unification
    ])

    preprocessing_pipeline.set_output(transform="pandas")

    return preprocessing_pipeline
     

## Test Preprocessing Pipeline

In [23]:
preprocessing = prepare_preprocessing()
preprocessing.fit(train_X, train_y).transform(train_X)

,Subsidy_Available,Current_Car_Type,Union_Subsidy_Available_Current_Car_Type,Charging_Stations_Near_Home,Gender,Union_Charging_Stations_Near_Home_Gender,Age,Union_Age_Current_Car_Type,Home_Charging_Possible,Union_Charging_Stations_Near_Home_Home_Charging_Possible_Age,...,Union_class_Annual_Income_USD_Current_Car_Type,Daily_Commute_km,class_Age,Annual_Income_USD,Environmental_Concern_Level,Union_Subsidy_Available_Environmental_Concern_Level,Union_Home_Charging_Possible_Subsidy_Available,Number_of_Cars_Owned,City_Type,Range_Anxiety_Level
0,No,Sedan,No Sedan,3,Male,3 Male,66,66 Sedan,Yes,3 Yes 66,...,3.0 Sedan,23.4,9.0,92887.0,1.0,No 1.0,Yes No,2,Suburban,Low
1,No,SUV,No SUV,2,Male,2 Male,38,38 SUV,Yes,2 Yes 38,...,0.0 SUV,5.0,2.0,30000.0,4.0,No 4.0,Yes No,1,Rural,Low
2,Yes,Sedan,Yes Sedan,8,Female,8 Female,26,26 Sedan,No,8 No 26,...,4.0 Sedan,36.8,0.0,94389.0,5.0,Yes 5.0,No Yes,1,Urban,Low
3,No,Hatchback,No Hatchback,6,Male,6 Male,66,66 Hatchback,Yes,6 Yes 66,...,2.0 Hatchback,23.7,9.0,73580.0,3.0,No 3.0,Yes No,2,Suburban,Low
4,No,Hatchback,No Hatchback,2,Male,2 Male,54,54 Hatchback,Yes,2 Yes 54,...,1.0 Hatchback,50.8,6.0,57898.0,3.0,No 3.0,Yes No,1,Suburban,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,Yes,Sedan,Yes Sedan,5,Male,5 Male,30,30 Sedan,No,5 No 30,...,2.0 Sedan,27.4,1.0,71090.0,5.0,Yes 5.0,No Yes,2,Urban,Medium
668661,Yes,SUV,Yes SUV,5,Female,5 Female,32,32 SUV,Yes,5 Yes 32,...,6.0 SUV,5.0,1.0,129990.0,1.0,Yes 1.0,Yes Yes,2,Suburban,Low
668662,Yes,Hatchback,Yes Hatchback,5,Female,5 Female,64,64 Hatchback,Yes,5 Yes 64,...,5.0 Hatchback,24.2,8.0,121791.0,1.0,Yes 1.0,Yes Yes,1,Suburban,Low
668663,Yes,SUV,Yes SUV,0,Female,0 Female,51,51 SUV,Yes,0 Yes 51,...,5.0 SUV,43.7,5.0,115923.0,1.0,Yes 1.0,Yes Yes,2,Rural,Low


# Explore mutual information 

In [24]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_selector


def prepare_dataset_for_mutual_info_classif(dataset):
    column_selector = make_column_selector(dtype_include=[object, "string"])
    ordinal_encoder = OrdinalEncoder()
    ordinal_column_transformer = (ordinal_encoder, column_selector)
    transformer = make_column_transformer(ordinal_column_transformer, remainder="passthrough", verbose_feature_names_out=False)
    transformer.set_output(transform="pandas")
    return transformer.fit_transform(dataset)


after_preprocessing = preprocessing.fit(train_X, train_y).transform(train_X)
dataset_for_mutual_info_classif = prepare_dataset_for_mutual_info_classif(after_preprocessing)

mutual_info = mutual_info_classif(dataset_for_mutual_info_classif, train_y)
result = pd.Series(mutual_info, index=after_preprocessing.columns)
result

Subsidy_Available                                                  0.161468
Current_Car_Type                                                   0.047602
Union_Subsidy_Available_Current_Car_Type                           0.102795
Charging_Stations_Near_Home                                        0.044436
Gender                                                             0.005655
Union_Charging_Stations_Near_Home_Gender                           0.002576
Age                                                                0.099446
Union_Age_Current_Car_Type                                         0.007297
Home_Charging_Possible                                             0.030744
Union_Charging_Stations_Near_Home_Home_Charging_Possible_Age       0.028606
class_Annual_Income_USD                                            0.033728
class_Daily_Commute_km                                             0.218493
Union_class_Annual_Income_USD_class_Daily_Commute_km               0.118772
class_availa

In [26]:
result.sort_values(ascending=False)

class_Daily_Commute_km                                             0.218493
Subsidy_Available                                                  0.161468
City_Type                                                          0.148605
Union_class_Annual_Income_USD_class_Daily_Commute_km               0.118772
Union_Subsidy_Available_Current_Car_Type                           0.102795
Age                                                                0.099446
Union_class_available_charging_stations_class_Annual_Income_USD    0.079303
Number_of_Cars_Owned                                               0.053967
class_available_charging_stations                                  0.050725
Current_Car_Type                                                   0.047602
Charging_Stations_Near_Home                                        0.044436
Union_class_Annual_Income_USD_Current_Car_Type                     0.043522
Range_Anxiety_Level                                                0.041557
class_Annual

# Create Model

## Prepare model Pipeline

In [27]:
def prepare_model_pipeline(preprocessing, model):
    preprocessing_step_label = "preprocessing"
    preprocessing_step = (preprocessing_step_label, preprocessing)

    model_step_label = "model"
    model_step = (model_step_label, model)

    steps = [
        preprocessing_step,
        model_step
    ]
    
    pipeline = Pipeline(steps)
    return pipeline

## Prepare StratifiedShuffleSplit

In [29]:
from sklearn.model_selection import StratifiedShuffleSplit

cross_validation_split = StratifiedShuffleSplit(n_splits=5, test_size=0.3)

## Evaluate model

In [36]:
from sklearn.model_selection import cross_val_score


def evaluate_model(model_pipeline, X, y, cross_validation_split=cross_validation_split, verbose_evaluation=False):
    metrics_scores = cross_val_score(model_pipeline, X, y, cv = cross_validation_split, scoring="roc_auc")
    return np.round(metrics_scores.mean(), 4).item()

## Using CatBoost

In [37]:
from catboost import CatBoostClassifier


class SafeCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return SafeCatBoostClassifier(**self.get_params())


preprocessing = prepare_preprocessing()
preprocessed_X = preprocessing.fit_transform(train_X, train_y)
category_features = make_column_selector(dtype_include=["string", object])(preprocessed_X)

model = SafeCatBoostClassifier(cat_features=category_features, verbose=False)
evaluate_model(model, preprocessed_X, train_y, verbose_evaluation=True)

0.9418

In [22]:
casual_category_features = make_column_selector(dtype_include=["string", object])(train_X)
casual_model = SafeCatBoostClassifier(cat_features=casual_category_features, verbose=False)
evaluate_model(casual_model, train_X, train_y, verbose_evaluation=True)

KeyboardInterrupt: 

In [38]:
model.fit(preprocessed_X, train_y)

SafeCatBoostClassifier(cat_features=['Subsidy_Available', 'Current_Car_Type', 'Union_Subsidy_Available_Current_Car_Type', 'Gender', 'Union_Charging_Stations_Near_Home_Gender', 'Union_Age_Current_Car_Type', 'Home_Charging_Possible', 'Union_Charging_Stations_Near_Home_Home_Charging_Possible_Age', 'Union_class_Annual_Income_USD_class_Daily_Commute_km', 'Union_class_available_charging_stations_class_Annual_Income_USD', 'Union_class_Annual_Income_USD_Current_Car_Type', 'Union_Subsidy_Available_Environmental_Concern_Level', 'Union_Home_Charging_Possible_Subsidy_Available', 'City_Type', 'Range_Anxiety_Level'], verbose=False)

In [40]:
test_x = test.drop(["id"], axis=1)

preprocessed_test_x = preprocessing.transform(test_x)
y_test = model.predict(preprocessed_test_x, prediction_type='Probability')

result = pd.DataFrame({"id": test["id"], "Will_Buy_EV": y_test[:, 1]})
result.to_csv("Predicting_Electric_Vehicle_Purchases_Submit_Prediction_Submission.csv", index=False)

In [33]:
from optuna import create_study

model = SafeCatBoostClassifier(cat_features=category_features, verbose=False)

def cat_boost_tuning_objective(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 100, 1000),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3),
        depth = trial.suggest_int("depth", 2, 8),
        cat_features=category_features,
        verbose=False
    )

    model.set_params(**params)
    accuracy = evaluate_model(model, preprocessed_X, train_y)
    return accuracy
    

study = create_study(study_name="cat_boost_tuning_objective", direction='maximize')
study.optimize(cat_boost_tuning_objective, n_trials=5)

best_params = study.best_params
model.set_params(**best_params)
evaluate_model(model, X, y, verbose_evaluation=True)

[I 2026-09-18 08:47:45,247] A new study created in memory with name: cat_boost_tuning_objective
[I 2026-09-18 09:13:20,520] Trial 0 finished with value: 0.9411 and parameters: {'n_estimators': 340, 'learning_rate': 0.2017600272721327, 'depth': 8}. Best is trial 0 with value: 0.9411.
[I 2026-09-18 09:29:24,251] Trial 1 finished with value: 0.9418 and parameters: {'n_estimators': 623, 'learning_rate': 0.29440448075108466, 'depth': 3}. Best is trial 1 with value: 0.9418.
[W 2026-09-18 09:43:23,584] Trial 2 failed with parameters: {'n_estimators': 368, 'learning_rate': 0.02743624051600646, 'depth': 7} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_58/2093923376.py", line 15, in cat_boost_tuning_objective
    accuracy = evaluate_model(model, preprocessed_X, tra

KeyboardInterrupt: 

In [ ]:
model.set_params(**best_params)
y_test = model.predict(preprocessed_test_x)

result = pd.DataFrame({"id": test["id"], "Will_Buy_EV": y_test})
result.to_csv("Predicting_Electric_Vehicle_Purchases_Submit_Prediction_Submission.csv", index=False)